# Tratamento dos Dados — Hospitais Habilitados em Oncologia

**Fontes:**
- `hospitais-habilitados-em-oncologia-abril-2025.xlsx`
- `hospitais-habilitados-em-reconstrucao-mamaria.xlsx`
- `tratamento-sincronico.xlsx`


---
## 1. Imports

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

---
## 2. Configurações Gerais

Ajuste aqui os caminhos de entrada/saída e a lista de colunas a remover antes de executar o pipeline.

In [2]:
# Caminhos
ENTRADA = Path("/home/anna/tcc/Limpeza e Integração/Hospitais Habilitados")
SAIDA = Path("/home/anna/tcc/Seleção e Transformação/Hospitais/tabela_hospitais_habilitados.csv")

# Arquivos de entrada
ARQUIVO_ONCOLOGIA = ENTRADA / "hospitais-habilitados-em-oncologia-abril-2025.xlsx"
ARQUIVO_RECONSTRUCAO = ENTRADA / "hospitais-habilitados-em-reconstrucao-mamaria.xlsx"
ARQUIVO_SINCRONICO = ENTRADA / "tratamento-sincronico.xlsx"

# Tabelas de municípios candidatas; usa a primeira que existir
CANDIDATOS_MUNICIPIOS = [
    Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv")
]

# Colunas a remover da tabela final
COLUNAS_REMOVER = [
    "UF",
    "CNES",
    "CÓDIGO",
    "CNPJ",
    "GESTÃO",
    "TIPO DE HABILITAÇÃO (CÓDIGO DO CNES)",
    "TIPO DE HABILITAÇÃO (Descrição)",
    "IBGE",
    "MACRORREGIÃO DE SAÚDE",
    "ESTABELECIMENTO",
    "HABILITAÇÃO",
]

SAIDA.parent.mkdir(parents=True, exist_ok=True)
print(f"✅ Pasta de saída: {SAIDA.parent}")

✅ Pasta de saída: /home/anna/tcc/Seleção e Transformação/Hospitais


---
## 3. Funções de Normalização de Texto

Funções para padronizar nomes de municípios antes do join com a tabela de códigos IBGE.
O pipeline de match usa três níveis:
1. `remover_acentos` — remove diacríticos
2. `normalizar_municipio` — strip + lowercase + sem acentos
3. `normalizar_chave_municipio` — normaliza pontuação e resolve aliases conhecidos

In [3]:
def remover_acentos(texto: str) -> str:
    """Remove todos os diacríticos (acentos, cedilha etc.) de uma string."""
    normalizado = unicodedata.normalize("NFD", texto)
    caracteres = []
    for caractere in normalizado:
        if unicodedata.category(caractere) != "Mn":
            caracteres.append(caractere)
    return "".join(caracteres)


def normalizar_municipio(valor: object) -> object:
    """Strip + lowercase + sem acentos. Retorna o valor original se não for string."""
    if isinstance(valor, str):
        return remover_acentos(valor.strip()).lower()
    return valor


def normalizar_chave_municipio(valor: object) -> object:
    """Normaliza o município para chave de junção (sem pontos e hífens) e resolve grafias alternativas."""
    if not isinstance(valor, str):
        return valor

    texto = normalizar_municipio(valor)
    texto = " ".join(str(texto).replace(".", " ").replace("-", " ").split())

    aliases = {
        "gov valadares": "governador valadares",
        "sao joao del rei": "sao joao del rei",
    }
    return aliases.get(texto, texto)


---
## 4. Carregamento dos Dados

### 4.1 Arquivos de hospitais habilitados

Três planilhas com estruturas distintas:
- **Oncologia:** cabeçalho na linha 2 (`header=1`), colunas 0-2 com forward-fill (células mescladas no Excel)
- **Reconstrução Mamária:** cabeçalho na linha 1 (`header=0`)
- **Tratamento Sincrônico:** cabeçalho na linha 2 (`header=1`), filtra linhas sem UF na coluna 1

In [4]:
def carregar_dados() -> tuple:
    """Lê as três planilhas brutas e retorna os DataFrames sem filtro de UF."""
    # Oncologia
    df_onco = pd.read_excel(ARQUIVO_ONCOLOGIA, header=1)
    df_onco.columns = df_onco.columns.str.strip()
    colunas_ffill = [df_onco.columns[0], df_onco.columns[1], df_onco.columns[2], df_onco.columns[3]]
    df_onco[colunas_ffill] = df_onco[colunas_ffill].ffill()
    df_onco = df_onco.dropna(subset=[df_onco.columns[4]]).reset_index(drop=True)

    # Reconstrução Mamária
    df_recon = pd.read_excel(ARQUIVO_RECONSTRUCAO, header=0)
    df_recon.columns = df_recon.columns.str.strip()

    # Tratamento Sincrônico
    df_sinc = pd.read_excel(ARQUIVO_SINCRONICO, header=1)
    df_sinc.columns = df_sinc.columns.str.strip()
    df_sinc = df_sinc[df_sinc[df_sinc.columns[1]].notna()].reset_index(drop=True)

    return df_onco, df_recon, df_sinc


df_onco, df_recon, df_sinc = carregar_dados()


---
## 5. Análise Inicial dos Dados

Inspeção rápida de cada planilha antes do filtro por UF.

In [5]:
for nome, df in [("Oncologia", df_onco), ("Reconstrução Mamária", df_recon), ("Tratamento Sincrônico", df_sinc)]:
    nulos = int(df.isnull().sum().sum())
    display(Markdown(f"**{nome}** — {df.shape[0]} linhas × {df.shape[1]} colunas | nulos totais: {nulos}"))
    display(df.head(3))

**Oncologia** — 370 linhas × 8 colunas | nulos totais: 19

,UF,MUNICÍPIO,MACRORREGIÃO DE SAÚDE,TOTAL DE HABILITAÇÕES,ESTABELECIMENTO,CNES,CÓDIGO,HABILITAÇÃO
0,AC,Rio Branco,Macro Única - AC,1,Hospital da Fundação Hospitalar Estadual do Acre,2001586,17.07,UNACON com Serviço de Radioterapia
1,AL,Arapiraca,2ª Macrorregião de Saúde,1,Complexo Hospitalar Manoel André - CHAMA,2005417,17.07,UNACON com Serviço de Radioterapia
2,AL,Maceió,1ª Macrorregião de Saúde,1,Hospital da Santa Casa de Misericórdia de Maceió,2007037,17.13,CACON com Serviço de Oncologia Pediátrica


**Reconstrução Mamária** — 165 linhas × 9 colunas | nulos totais: 33

,UF,IBGE,MUNICÍPIO,ESTABELECIMENTO,CNES,CNPJ,GESTÃO,TIPO DE HABILITAÇÃO (CÓDIGO DO CNES),TIPO DE HABILITAÇÃO (Descrição)
0,BA,292740,SALVADOR,HOSPITAL ARISTIDES MALTEZ,0003786,15.180.961/0001-00,MUNICIPAL,17.23,Reconstrução Mamária Pós-Mastectomia Total
1,BA,290000,SALVADOR,HOSPITAL UNIVERSITÁRIO PROFESSOR EDGARD SANTOS,0003816,15.180.714/0002-87,ESTADUAL,17.23,Reconstrução Mamária Pós-Mastectomia Total
2,BA,293135,TEIXEIRA DE FREITAS,HOSPITAL MUNICIPAL DE TEIXEIRA DE FREITAS,2301318,13.650.403/0001-28,MUNICIPAL,17.23,Reconstrução Mamária Pós-Mastectomia Total


**Tratamento Sincrônico** — 15 linhas × 8 colunas | nulos totais: 0

,TOTAL DE HABILITAÇÕES,UF,MUNICÍPIO,MACRORREGIÃO DE SAÚDE,ESTABELECIMENTO,CNES,CÓDIGO,HABILITAÇÃO
0,1,BA,Salvador,Leste,Hospital Aristides Maltez,3786.0,17.22,TRATAMENTOS INTEGRADOS SINCRONICOS EM ONCOLOGIA
1,1,BA,Vitória da Conquista,Sudoeste,Hospital Geral de Vitória da Conquista,2402076.0,17.22,TRATAMENTOS INTEGRADOS SINCRONICOS EM ONCOLOGIA
2,1,CE,Fortaleza,1ª Macro-Fortaleza,Hospital Haroldo Juaçaba,2723220.0,17.22,TRATAMENTOS INTEGRADOS SINCRONICOS EM ONCOLOGIA


---
## 6. Filtro por Estado (MG) e União das Tabelas

Filtra apenas registros de Minas Gerais e une os três DataFrames em uma única tabela,
adicionando a coluna `TIPO` para identificar a origem de cada linha.

In [6]:
def filtrar_mg(df_onco, df_recon, df_sinc) -> tuple:
    """Filtra cada DataFrame mantendo apenas linhas onde a coluna de UF == 'MG'."""
    col_onco = df_onco.columns[0]
    col_recon = df_recon.columns[0]
    col_sinc = df_sinc.columns[1]  # no Sincrônico a UF está na coluna 1

    df_onco_mg = df_onco[df_onco[col_onco].astype(str).str.strip() == "MG"].reset_index(drop=True)
    df_recon_mg = df_recon[df_recon[col_recon].astype(str).str.strip() == "MG"].reset_index(drop=True)
    df_sinc_mg = df_sinc[df_sinc[col_sinc].astype(str).str.strip() == "MG"].reset_index(drop=True)

    return df_onco_mg, df_recon_mg, df_sinc_mg


def unir_tabelas(df_onco_mg, df_recon_mg, df_sinc_mg) -> pd.DataFrame:
    """Concatena os três DataFrames com rótulo de tipo na coluna 'TIPO'."""
    tabela = pd.concat(
        {
            "Oncologia": df_onco_mg,
            "Reconstrução Mamária": df_recon_mg,
            "Tratamento Sincrônico": df_sinc_mg,
        },
        axis=0, sort=False,
    )
    tabela.index.names = ["TIPO", "ID_ORIGINAL"]
    return tabela.reset_index(level="TIPO")


df_onco_mg, df_recon_mg, df_sinc_mg = filtrar_mg(df_onco, df_recon, df_sinc)
tabela_unida = unir_tabelas(df_onco_mg, df_recon_mg, df_sinc_mg)

print("╔══════════════════════════════════════════════════╗")
print("║            Após filtro MG                        ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Oncologia            : {len(df_onco_mg):<23} ║")
print(f"║  Reconstrução Mamária : {len(df_recon_mg):<23} ║")
print(f"║  Tratamento Sincrônico: {len(df_sinc_mg):<23} ║")
print(f"║  Total unido          : {len(tabela_unida):<23} ║")
print("╚══════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════╗
║            Após filtro MG                        ║
╠══════════════════════════════════════════════════╣
║  Oncologia            : 40                      ║
║  Reconstrução Mamária : 24                      ║
║  Tratamento Sincrônico: 1                       ║
║  Total unido          : 65                      ║
╚══════════════════════════════════════════════════╝


---
## 7. Enriquecimento com Código IBGE

Carrega a tabela de municípios × código IBGE e faz o join pelo nome normalizado do município.
Municípios sem correspondência são listados para revisão manual.

In [7]:
def carregar_mapa_codigo_municipio() -> dict:
    """Carrega o mapa MUNICÍPIO NORMALIZADO -> CÓDIGO IBGE do primeiro arquivo existente de CANDIDATOS_MUNICIPIOS."""
    arquivo = None
    for candidato in CANDIDATOS_MUNICIPIOS:
        if candidato.exists():
            arquivo = candidato
            break
    if arquivo is None:
        candidatos = ", ".join(str(candidato) for candidato in CANDIDATOS_MUNICIPIOS)
        raise FileNotFoundError(f"Nenhum arquivo de municípios encontrado. Tentativas: {candidatos}")

    df = pd.read_csv(arquivo, encoding="utf-8-sig")
    obrigatorias = {"CÓDIGO IBGE", "MUNICÍPIO NORMALIZADO"}
    faltando = obrigatorias - set(df.columns)
    if faltando:
        raise ValueError(f"Arquivo de municípios sem colunas obrigatórias: {sorted(faltando)}")

    df["municipio_chave"] = df["MUNICÍPIO NORMALIZADO"].map(normalizar_chave_municipio)
    df = df.dropna(subset=["municipio_chave", "CÓDIGO IBGE"])

    return dict(zip(df["municipio_chave"].astype(str), df["CÓDIGO IBGE"].astype(int)))


def gerar_tabela_habilitados(tabela: pd.DataFrame) -> pd.DataFrame:
    """Limpa a tabela unida, acrescenta o código IBGE e 2023_TOTAL e colapsa para uma linha por (município, código, tipo)."""
    # Detecta coluna de município
    if "MUNICÍPIO" in tabela.columns:
        col_municipio = "MUNICÍPIO"
    elif "MUNICÍPIOS" in tabela.columns:
        col_municipio = "MUNICÍPIOS"
    else:
        raise ValueError(f"Coluna de município não encontrada. Colunas disponíveis: {tabela.columns.tolist()}")

    mapa_codigo = carregar_mapa_codigo_municipio()
    df = tabela.copy()

    # Remove colunas desnecessárias
    colunas_existentes = []
    for coluna in COLUNAS_REMOVER:
        if coluna in df.columns:
            colunas_existentes.append(coluna)
    df = df.drop(columns=colunas_existentes)

    # Normaliza e faz join com código IBGE
    df[col_municipio] = df[col_municipio].map(normalizar_municipio)
    df["municipio_chave"] = df[col_municipio].map(normalizar_chave_municipio)
    df["CÓDIGO IBGE"]     = df["municipio_chave"].map(mapa_codigo)
    df = df.drop(columns=["municipio_chave"])

    # Padroniza nome da coluna de município
    if col_municipio != "MUNICÍPIO":
        df = df.rename(columns={col_municipio: "MUNICÍPIO"})

    # Converte código para Int64
    df["CÓDIGO IBGE"] = pd.to_numeric(df["CÓDIGO IBGE"], errors="coerce").astype("Int64")

    # Padroniza categorias de TIPO em CAIXA ALTA
    df["TIPO"] = df["TIPO"].str.upper()

    # Cada linha é uma habilitação; casos sem valor de origem contam como 1
    df["TOTAL DE HABILITAÇÕES"] = pd.to_numeric(df["TOTAL DE HABILITAÇÕES"], errors="coerce").fillna(1).astype(int)

    # 2023_TOTAL: soma de habilitações por município e tipo, no lugar da coluna bruta
    df["2023_TOTAL"] = df.groupby(["MUNICÍPIO", "TIPO"])["TOTAL DE HABILITAÇÕES"].transform("sum")
    df = df.drop(columns=["TOTAL DE HABILITAÇÕES"])

    # Uma linha por (município, tipo), para a soma por regional não ser inflada
    df = df.drop_duplicates(subset=["MUNICÍPIO", "CÓDIGO IBGE", "TIPO"]).reset_index(drop=True)

    # Reordena colunas
    colunas_fixas = ["MUNICÍPIO", "CÓDIGO IBGE"]
    colunas_demais = []
    for coluna in df.columns:
        if coluna not in colunas_fixas:
            colunas_demais.append(coluna)
    return df[colunas_fixas + colunas_demais]


df_tabela = gerar_tabela_habilitados(tabela_unida)

# Relatório de municípios sem código
sem_codigo = df_tabela[df_tabela["CÓDIGO IBGE"].isna()]["MUNICÍPIO"].dropna().unique().tolist()
if sem_codigo:
    display(Markdown(f"⚠️ **{len(sem_codigo)} município(s) sem correspondência de código IBGE:**"))
    for municipio in sorted(sem_codigo):
        print(f"  - {municipio}")
else:
    print("Todos os municípios tiveram código IBGE encontrado.")

print(f"\nShape final: {df_tabela.shape}")

Todos os municípios tiveram código IBGE encontrado.

Shape final: (43, 4)


---
## 7.1 Regional de Saúde

Adiciona `REGIONAL DE SAÚDE` via join pelo Código IBGE com a mesma tabela de municípios.

In [8]:
def carregar_mapa_regional() -> dict:
    """Carrega o mapeamento CÓDIGO IBGE → REGIONAL DE SAÚDE a partir da tabela de municípios."""
    arquivo = None
    for candidato in CANDIDATOS_MUNICIPIOS:
        if candidato.exists():
            arquivo = candidato
            break
    if arquivo is None:
        candidatos = ", ".join(str(candidato) for candidato in CANDIDATOS_MUNICIPIOS)
        raise FileNotFoundError(f"Nenhum arquivo de municípios encontrado. Tentativas: {candidatos}")

    df = pd.read_csv(arquivo, encoding="utf-8-sig")
    df["CÓDIGO IBGE"] = pd.to_numeric(df["CÓDIGO IBGE"], errors="coerce")
    df = df.dropna(subset=["CÓDIGO IBGE"])
    return dict(zip(df["CÓDIGO IBGE"].astype(int), df["REGIONAL NORMALIZADA"]))


mapa_regional = carregar_mapa_regional()
df_tabela["REGIONAL DE SAÚDE"] = df_tabela["CÓDIGO IBGE"].map(mapa_regional)

sem_regional = int(df_tabela["REGIONAL DE SAÚDE"].isna().sum())
if sem_regional > 0:
    print(f"⚠️ {sem_regional} linha(s) sem Regional de Saúde correspondente.")
else:
    print("Todas as linhas têm Regional de Saúde correspondente.")

# Reordena: MUNICÍPIO, CÓDIGO IBGE, REGIONAL DE SAÚDE, restante
colunas = df_tabela.columns.tolist()
colunas.remove("REGIONAL DE SAÚDE")
posicao = colunas.index("CÓDIGO IBGE") + 1
colunas.insert(posicao, "REGIONAL DE SAÚDE")
df_tabela = df_tabela[colunas]

Todas as linhas têm Regional de Saúde correspondente.


---
## 8. Inspeção Pós-Processamento

Verificação da tabela final antes de salvar: distribuição por tipo, municípios únicos e valores ausentes.

In [9]:
display(Markdown("**Distribuição por tipo de habilitação:**"))
display(
    df_tabela["TIPO"].value_counts().rename_axis("Tipo").reset_index(name="Qtd")
    .style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    .hide(axis="index")
)

display(Markdown(f"**Municípios únicos com código IBGE identificado:** {df_tabela['CÓDIGO IBGE'].notna().sum()} / {len(df_tabela)}"))

ausentes = df_tabela.isnull().sum()
ausentes = ausentes[ausentes > 0]
if not ausentes.empty:
    display(Markdown("**Valores ausentes por coluna:**"))
    display(ausentes.to_frame("Nulos").style.background_gradient(cmap="Reds"))
else:
    display(Markdown("✅ **Sem valores ausentes nas colunas principais.**"))

display(Markdown("**Primeiras linhas da tabela final:**"))
display(df_tabela.head(5))

**Distribuição por tipo de habilitação:**

Tipo,Qtd
ONCOLOGIA,25
RECONSTRUÇÃO MAMÁRIA,17
TRATAMENTO SINCRÔNICO,1


**Municípios únicos com código IBGE identificado:** 43 / 43

✅ **Sem valores ausentes nas colunas principais.**

**Primeiras linhas da tabela final:**

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,TIPO,2023_TOTAL
0,belo horizonte,3106200,belo horizonte,ONCOLOGIA,7
1,betim,3106705,belo horizonte,ONCOLOGIA,1
2,curvelo,3120904,sete lagoas,ONCOLOGIA,1
3,itabira,3131703,itabira,ONCOLOGIA,1
4,sete lagoas,3167202,sete lagoas,ONCOLOGIA,1


---
## 9. Salvando o Resultado


In [10]:
df_tabela.to_csv(SAIDA, index=False, encoding="utf-8-sig")